## Load spatial references points

In [254]:
import pandas as pd
import os
import numpy as np
from Sequential_Fish.tools.utils import open_image

REF_CHANNEL = 0
CORR_CHANNEL = 1
DEGREE = 2
MAX_DISTANCE = 2000
VOXEL_SIZE = 200,97,97
VOXEL_SIZE = np.array([VOXEL_SIZE])


image_location = "/media/SSD_floricslimani/Fish_seq/Davide/Chromatic abberations/"
filename = "Beads-Field_01"
channels = [0,1,2]

In [255]:
df = pd.DataFrame()
for chan in channels :
    coordinates = pd.read_csv(image_location + f"{filename}_channel{chan}.csv", sep=";")
    coordinates.loc[:,['channel']] = chan
    df = pd.concat([
        df,
        coordinates,
    ], axis=0)
df = df.loc[:,['coordinates', 'channel']]
df

,coordinates,channel
0,"(23, 90, 1872)",0
1,"(23, 108, 1884)",0
2,"(23, 1918, 4)",0
3,"(24, 62, 1764)",0
4,"(24, 80, 1734)",0
...,...,...
513,"(43, 338, 1250)",2
514,"(43, 1146, 1605)",2
515,"(45, 1488, 942)",2
516,"(46, 923, 1042)",2


In [256]:
df['new_coordinates'] = df['coordinates'].str.replace("(","").str.replace(")","").str.split(',')
df['z_nm'] = df['new_coordinates'].apply(lambda x : int(x[0])*VOXEL_SIZE[0,0])
df['y_nm'] = df['new_coordinates'].apply(lambda x : int(x[1])*VOXEL_SIZE[0,1])
df['x_nm'] = df['new_coordinates'].apply(lambda x : int(x[2])*VOXEL_SIZE[0,2])

df['z'] = df['new_coordinates'].apply(lambda x : int(x[0]))
df['y'] = df['new_coordinates'].apply(lambda x : int(x[1]))
df['x'] = df['new_coordinates'].apply(lambda x : int(x[2]))


df['coordinates_nm'] = list(zip(df['z_nm'], df['y_nm'], df['x_nm'],))
df['coordinates'] = list(zip(df['z'], df['y'], df['x'],))
df = df.drop("new_coordinates", axis=1)
df

,coordinates,channel,z_nm,y_nm,x_nm,z,y,x,coordinates_nm
0,"(23, 90, 1872)",0,4600,8730,181584,23,90,1872,"(4600, 8730, 181584)"
1,"(23, 108, 1884)",0,4600,10476,182748,23,108,1884,"(4600, 10476, 182748)"
2,"(23, 1918, 4)",0,4600,186046,388,23,1918,4,"(4600, 186046, 388)"
3,"(24, 62, 1764)",0,4800,6014,171108,24,62,1764,"(4800, 6014, 171108)"
4,"(24, 80, 1734)",0,4800,7760,168198,24,80,1734,"(4800, 7760, 168198)"
...,...,...,...,...,...,...,...,...,...
513,"(43, 338, 1250)",2,8600,32786,121250,43,338,1250,"(8600, 32786, 121250)"
514,"(43, 1146, 1605)",2,8600,111162,155685,43,1146,1605,"(8600, 111162, 155685)"
515,"(45, 1488, 942)",2,9000,144336,91374,45,1488,942,"(9000, 144336, 91374)"
516,"(46, 923, 1042)",2,9200,89531,101074,46,923,1042,"(9200, 89531, 101074)"


In [257]:
coordinates = {chan : df.loc[df['channel'] == chan,['z','y','x']].to_numpy().astype(int) for chan in channels }
coordinates_nm = {chan : df.loc[df['channel'] == chan,['z_nm','y_nm','x_nm']].to_numpy().astype(int) for chan in channels }

## Match beads

In [258]:
from sklearn.neighbors import NearestNeighbors
def match_beads(coords1, coords2, max_dist=5):
    """Match nearest beads between two channels."""
    nn = NearestNeighbors(n_neighbors=1).fit(coords2)
    distances, indices = nn.kneighbors(coords1)
    matches = distances[:, 0] < max_dist
    return coords1[matches], coords2[indices[matches, 0]]

In [259]:
coords, dist = match_beads(
    coordinates_nm[CORR_CHANNEL],
    coordinates_nm[REF_CHANNEL],
    max_dist=MAX_DISTANCE
    )

print("total spot : ", len(coordinates[CORR_CHANNEL]))
print("total spot_ref : ", len(coordinates[REF_CHANNEL]))
print("match found : ", len(coords))

total spot :  532
total spot_ref :  519
match found :  525


## Fit polynomial

In [260]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy.ndimage import map_coordinates

def fit_polynomial_transform_3d(src_points, dst_points, degree=2):
    """Fit 3D polynomial regression mapping coords → dst."""
    poly = PolynomialFeatures(degree)
    X_poly = poly.fit_transform(src_points)
    model_x = LinearRegression().fit(X_poly, dst_points[:, 2])  # x
    model_y = LinearRegression().fit(X_poly, dst_points[:, 1])  # y
    model_z = LinearRegression().fit(X_poly, dst_points[:, 0])  # z
    return poly, model_x, model_y, model_z

In [261]:
poly, model_x, model_y, model_z = fit_polynomial_transform_3d(
    dist, 
    coords,
    degree=DEGREE
)

## Correct abberations

In [262]:
def apply_polynomial_transform_3d(volume, poly, model_x, model_y, model_z):
    """Warp 3D volume using learned polynomial transform."""
    z, y, x = volume.shape
    zz, yy, xx = np.meshgrid(np.arange(z), np.arange(y), np.arange(x), indexing='ij')
    coords = np.stack([zz.ravel(), yy.ravel(), xx.ravel()], axis=1)

    X_poly = poly.transform(coords * VOXEL_SIZE)
    new_z_nm = model_z.predict(X_poly)
    new_y_nm = model_y.predict(X_poly)
    new_x_nm = model_x.predict(X_poly)

    #convert back to pixel
    new_coords_pixel = np.stack([new_z_nm, new_y_nm, new_x_nm], axis=0) / VOXEL_SIZE.T

    warped = map_coordinates(volume, new_coords_pixel, order=1, mode='reflect').reshape(z, y, x)
    return warped

In [263]:
image = open_image(image_location + "/Beads-Field_01.tif")
image.shape

(47, 1981, 2004, 3)

In [264]:
image_corrected = apply_polynomial_transform_3d(
    image[...,CORR_CHANNEL],
    poly=poly,
    model_x=model_x,
    model_y=model_y,
    model_z=model_z
)
image_corrected.shape

(47, 1981, 2004)

## Widget

In [ ]:
import napari
from abc import ABC, abstractmethod
from magicgui import magicgui
from magicgui import widgets
from napari.layers import Points, Image, Layer
from typing import Tuple
from bigfish.detection import detect_spots

In [ ]:
class NapariWidget(ABC) :
    """
    Common super class for custom widgets added to napari interface during run
    Each sub class as a specific function, but the widget can be acess with attribute .widget
    """
    def __init__(self):
        self.widget = self._create_widget()

    @abstractmethod
    def _create_widget(self) :
        """
        This should return a widget you can add to the napari (QWidget)
        """
        pass

In [309]:
class BeadsDetector(NapariWidget) :
    def __init__(self):
        super().__init__()

    def _create_widget(self):

        @magicgui(
            beads_image = {'label' : 'Select image layer for beads detection :'},
            threshold = {'label' : 'Threshold', 'min': 1},
            voxel_size = {'annotation' : Tuple[int,int,int], 'label' : 'Voxel size (zyx)'},
            beads_radius = {'annotation' : Tuple[int,int,int], 'label' : 'Beads radius (zyx)'}
                
        )
        def detect_beads(
            beads_image : Image,
            voxel_size : Tuple[int,int,int],
            beads_radius : Tuple[int,int,int],
            threshold : int
        ) -> Points :

            print("hello")

            coordinates = detect_spots(
                beads_image.data,
                threshold=threshold,
                voxel_size=voxel_size,
                spot_radius=beads_radius
            )

            print("coordinates : ", coordinates.shape, name='detectd_beads')

            return Points(data=coordinates, ndim=len(voxel_size))
        return detect_beads

In [ ]:
class ChromaticAberrationCorector(NapariWidget) :
    def __init__(self):
        super().__init__()

    def _create_widget(self):

        @magicgui(
                reference_image={'label' : 'Select reference image :'},
                spatial_reference={'label' : 'Select points reference for reference image :'},
                spatial_reference_shifted={'label' : 'Select points reference for image to correct :'},
                voxel_size = {'annotation' :Tuple[int,int,int], 'label' : "Voxel size (zyx)"},
                auto_call=False,
                call_button= "Correct chromatic aberrations",
        )
        def create_corrected_layer(
            reference_image : Image,
            spatial_reference : Points,
            spatial_reference_shifted : Points,
            voxel_size : Tuple[int,int,int],
        ) ->  Image :
            pass


        return create_corrected_layer
    


## Napari

In [308]:
os.environ["QT_QPA_PLATFORM"] = "xcb"
Viewer = napari.Viewer()
Viewer.add_image(image[...,REF_CHANNEL], name= "ref", scale=(200,97,97), blending='additive', colormap = 'green', interpolation2d= 'cubic')
Viewer.add_image(image[...,CORR_CHANNEL], name= "uncorrected", scale=(200,97,97), blending='additive', colormap = 'red', interpolation2d= 'cubic')
Viewer.add_image(image_corrected, name= "corrected", scale=(200,97,97), blending='additive', colormap = 'blue', visible=False, interpolation2d= 'cubic')

#Widgets
beads_detector = BeadsDetector()
abberration_corrector = ChromaticAberrationCorector()
right_container = widgets.Container(widgets=[beads_detector.widget], labels=False)
left_container = widgets.Container(widgets=[abberration_corrector.widget], labels=False)
Viewer.window.add_dock_widget(left_container, name='', area='left')
Viewer.window.add_dock_widget(right_container, name='', area='right')

pass